# JOIN TIGER BLOCK GROUPS WITH POPULATION

In [2]:
import pandas as pd
import arcpy

# ── 1. FILE PATHS ────────────────────────────────────────────────────────────
TIGER_SHP = r"D:\GIS_Seminar_Project\PC_TIGER_Data\PC_TIGER_BlockGroups.shp"
CENSUS_CSV = r"D:\GIS_Seminar_Project\PC_Census_Block_Groups\Census.csv"
OUTPUT_SHP = r"D:\GIS_Seminar_Project\PC_TIGER_Data\PC_BlockGroups_Population.shp"

# ── 2. LOAD CENSUS CSV & CLEAN UP THE GEO_ID ─────────────────────────────────
# The Census GEO_ID looks like: "1500000US121030201051"
# The TIGER GEOID looks like:   "121030201051"
# We strip the "1500000US" prefix so both sides match.

census = pd.read_csv(CENSUS_CSV, skiprows=[1])  # row 0 = real header (kept),
                                                # row 1 = description row (skipped),
                                                # [1] means skip only that one row

census = census.rename(columns={
    "GEO_ID":       "GEO_ID",
    "B01003_001E":  "POPULATION"
})

# Strip the prefix and keep only the 12-digit GEOID
census["GEOID"] = census["GEO_ID"].str.replace("1500000US", "", regex=False)

# Keep only the two columns we need
census = census[["GEOID", "POPULATION"]]

# Make sure population is numeric
census["POPULATION"] = pd.to_numeric(census["POPULATION"], errors="coerce")

print(f"Census rows loaded: {len(census)}")
print(census.head(3))

# ── 3. SAVE CLEANED CENSUS TABLE AS A TEMP CSV FOR ARCPY JOIN ────────────────
TEMP_CSV = r"D:\GIS_Seminar_Project\PC_Census_Block_Groups\Census_clean.csv"
census.to_csv(TEMP_CSV, index=False)
print(f"Cleaned CSV saved to: {TEMP_CSV}")

# ── 4. COPY THE SHAPEFILE SO WE DON'T MODIFY THE ORIGINAL ───────────────────
arcpy.management.CopyFeatures(TIGER_SHP, OUTPUT_SHP)
print("Shapefile copied.")

# ── 5. JOIN POPULATION TO THE COPIED SHAPEFILE ───────────────────────────────
# JoinField permanently adds POPULATION column to the output shapefile.
# "GEOID"  = field in the shapefile
# "GEOID"  = field in the cleaned CSV (same name after our rename above)

arcpy.management.JoinField(
    in_data        = OUTPUT_SHP,
    in_field       = "GEOID",        # field in TIGER shapefile
    join_table     = TEMP_CSV,
    join_field     = "GEOID",        # field in cleaned census CSV
    fields         = ["POPULATION"]  # only bring in the population column
)
print("Join complete.")

# ── 6. QUICK CHECK ───────────────────────────────────────────────────────────
result = arcpy.management.GetCount(OUTPUT_SHP)
print(f"Total block groups in output: {result[0]}")

# Print first 5 rows to confirm population joined correctly
fields = ["GEOID", "POPULATION"]
with arcpy.da.SearchCursor(OUTPUT_SHP, fields) as cursor:
    print("\nSample rows:")
    for i, row in enumerate(cursor):
        print(f"  GEOID: {row[0]}  |  Population: {row[1]}")
        if i == 4:
            break

print("\nDone! Output saved to:", OUTPUT_SHP)

Census rows loaded: 737
          GEOID  POPULATION
0  121030201051        1188
1  121030201052        2980
2  121030201053        1818
Cleaned CSV saved to: D:\GIS_Seminar_Project\PC_Census_Block_Groups\Census_clean.csv
Shapefile copied.
Join complete.
Total block groups in output: 745

Sample rows:
  GEOID: 121030248041  |  Population: 0
  GEOID: 121030269132  |  Population: 0
  GEOID: 121030254171  |  Population: 0
  GEOID: 121030253092  |  Population: 0
  GEOID: 121030269164  |  Population: 0

Done! Output saved to: D:\GIS_Seminar_Project\PC_TIGER_Data\PC_BlockGroups_Population.shp


In [8]:
import pandas as pd
import struct
import arcpy

# ── 1. FILE PATHS ────────────────────────────────────────────────────────────
TIGER_SHP  = r"D:\GIS_Seminar_Project\PC_TIGER_Data\PC_TIGER_BlockGroups.shp"
TIGER_DBF  = r"D:\GIS_Seminar_Project\PC_TIGER_Data\PC_TIGER_BlockGroups.dbf"
CENSUS_CSV = r"D:\GIS_Seminar_Project\PC_Census_Block_Groups\Census.csv"
OUTPUT_SHP = r"D:\GIS_Seminar_Project\PC_TIGER_Data\PC_BlockGroups_Population.shp"
TEMP_CSV   = r"D:\GIS_Seminar_Project\PC_TIGER_Data\Census_clean.csv"

# ── 2. EXTRACT REAL GEOIDs DIRECTLY FROM DBF RAW BYTES ───────────────────────
# ArcGIS misreads this DBF — the real 12-digit GEOID sits at byte offset 1.
# We read it directly and build a list in DBF row order.

tiger_geoids = []
with open(TIGER_DBF, 'rb') as f:
    f.read(4)
    num_records = struct.unpack('<I', f.read(4))[0]
    header_size = struct.unpack('<H', f.read(2))[0]
    record_size = struct.unpack('<H', f.read(2))[0]
    f.seek(header_size)                        # jump straight to first record
    for i in range(num_records):
        rec = f.read(record_size)
        geoid = rec[1:13].decode('latin-1', 'ignore').strip()
        tiger_geoids.append(geoid)

print(f"TIGER GEOIDs extracted: {len(tiger_geoids)}")
print("Sample:", tiger_geoids[:3])

# ── 3. LOAD & CLEAN CENSUS CSV ───────────────────────────────────────────────
census = pd.read_csv(CENSUS_CSV, skiprows=[1])
census['GEOID_JOIN'] = census['GEO_ID'].str.replace('1500000US', '', regex=False)
census['POPULATION'] = pd.to_numeric(census['B01003_001E'], errors='coerce').fillna(0).astype(int)
census = census[['GEOID_JOIN', 'POPULATION']]
pop_lookup = dict(zip(census['GEOID_JOIN'], census['POPULATION']))

print(f"Census rows loaded: {len(census)}")

# ── 4. COPY SHAPEFILE ────────────────────────────────────────────────────────
if arcpy.Exists(OUTPUT_SHP):
    arcpy.management.Delete(OUTPUT_SHP)
arcpy.management.CopyFeatures(TIGER_SHP, OUTPUT_SHP)
print("Shapefile copied.")

# ── 5. ADD FIELDS ────────────────────────────────────────────────────────────
arcpy.management.AddField(OUTPUT_SHP, "GEOID_JOIN", "TEXT", field_length=12)
arcpy.management.AddField(OUTPUT_SHP, "POPULATION", "LONG")

# ── 6. WRITE GEOID_JOIN FIRST USING GEOIDFQ AS THE ANCHOR ───────────────────
# GEOIDFQ is readable by ArcGIS and contains the full qualified ID
# e.g. "1500000US121030248041" — we strip the prefix to get our GEOID_JOIN.
# This avoids any OID ordering assumption entirely.

with arcpy.da.UpdateCursor(OUTPUT_SHP, ["GEOIDFQ", "GEOID_JOIN", "POPULATION"]) as cursor:
    for row in cursor:
        geoidfq = str(row[0]).strip() if row[0] else ""
        # Strip "1500000US" prefix → 12-digit GEOID
        geoid = geoidfq.replace("1500000US", "").strip()[-12:] if geoidfq else ""
        population = pop_lookup.get(geoid, 0)
        row[1] = geoid
        row[2] = population
        cursor.updateRow(row)

# ── 7. VERIFY ────────────────────────────────────────────────────────────────
matched = 0
print("\nSample rows after join:")
with arcpy.da.SearchCursor(OUTPUT_SHP, ["GEOIDFQ", "GEOID_JOIN", "POPULATION"]) as cursor:
    for i, row in enumerate(cursor):
        if row[2] and row[2] > 0:
            matched += 1
        if i < 5:
            print(f"  GEOIDFQ: {row[0]}  |  GEOID_JOIN: {row[1]}  |  Population: {row[2]}")

print(f"\nBlock groups with population > 0: {matched} of 737")
print(f"Done! Output: {OUTPUT_SHP}")

TIGER GEOIDs extracted: 737
Sample: ['121030248041', '121030269132', '121030254171']
Census rows loaded: 737
Shapefile copied.

Sample rows after join:
  GEOIDFQ: 1500000US121030248041  |  GEOID_JOIN: 121030248041  |  Population: 2802
  GEOIDFQ: 1500000US121030269132  |  GEOID_JOIN: 121030269132  |  Population: 1553
  GEOIDFQ: 1500000US121030254171  |  GEOID_JOIN: 121030254171  |  Population: 802
  GEOIDFQ: 1500000US121030253092  |  GEOID_JOIN: 121030253092  |  Population: 1362
  GEOIDFQ: 1500000US121030269164  |  GEOID_JOIN: 121030269164  |  Population: 986

Block groups with population > 0: 733 of 737
Done! Output: D:\GIS_Seminar_Project\PC_TIGER_Data\PC_BlockGroups_Population.shp
